# 🌿 Análisis Exploratorio y Validación Ambiental - Green Data ETL
## Integración: Inventario Nacional GEI (1990-2024) + API Calidad del Aire (WAQI)

Este notebook permite explorar los datos consolidados en las capas **Bronze, Silver y Gold** tras la ejecución del pipeline ETL.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Conexión a la base de datos Gold en SQLite
db_path = Path("../data/gold/green_data_warehouse.db")
conn = sqlite3.connect(db_path)

# Listar tablas en el Data Warehouse
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tablas disponibles en la capa Gold:")
print(tables)

In [ ]:
# 1. Resumen de KPIs Ejecutivos
df_kpis = pd.read_sql_query("SELECT * FROM mart_kpis_resumen_ejecutivo;", conn)
df_kpis

In [ ]:
# 2. Evolución Histórica de Emisiones Netas vs Absorciones Forestales por Año (1990 - 2024)
query_hist = """
SELECT 
    anio,
    SUM(emisiones_brutas_kt) AS total_bruto_kt,
    SUM(absorciones_kt) AS total_absorciones_kt,
    SUM(emisiones_netas_kt) AS total_neto_kt
FROM mart_consolidado_ambiental
GROUP BY anio
ORDER BY anio;
"""
df_hist = pd.read_sql_query(query_hist, conn)
df_hist.head(10)

In [ ]:
# 3. Monitoreo en Tiempo Real de Calidad del Aire (Red WAQI)
query_air = """
SELECT 
    e.estacion_nombre,
    e.ciudad,
    f.aqi,
    f.aqi_categoria,
    f.pm25,
    f.pm10,
    f.temperatura_c
FROM fact_calidad_aire_tiempo_real f
JOIN dim_estacion_monitoreo e ON f.estacion_id = e.estacion_id
ORDER BY f.aqi DESC;
"""
df_air = pd.read_sql_query(query_air, conn)
df_air